In [2]:
import openai
import re
import time
import json

import numpy as np

from tqdm import tqdm
from pprint import pprint
from tenacity import retry, stop_after_attempt, wait_chain, wait_fixed

import os
from openai import AzureOpenAI

import math

import re
import math
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
import traceback

In [3]:
endpoint = "https://pankajaiml.openai.azure.com/"
model_name = "gpt-35-turbo"
deployment = "gpt-35-turbo"
subscription_key = "REDACTED_AZURE_OPENAI_KEY"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Retry logic
@retry(wait=wait_chain(*[wait_fixed(3) for _ in range(3)] +
                       [wait_fixed(5) for _ in range(2)] +
                       [wait_fixed(10)]))
def completion_with_backoff(messages):
    return client.chat.completions.create(
        messages=messages,
        max_tokens=1024,
        temperature=0.0,
        model=deployment
    )

In [4]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as reader:
        data = json.load(reader)  # Load the entire JSON file
    return data

dev_data = load_json('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/testingDatasets/AQuAsampled_train.json')
CoT_prompt_examples = open('/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/CoT_prompt_examples.txt').read()
Standard_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/standard_prompt_examples.txt").read()
CCoT_prompt_examples = open("/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/prompt_examples/CCoT_prompt_example.txt").read()

In [4]:
# === Metrics ===
acc = 0
total = 0

# === File Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/AQuA/CoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return float(cleaned)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        question = d['question']
        options = d['options']
        correct_choice = d['correct'].strip().upper()

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}"

        # === Prompt Setup ===
        prompt_q = (
            CoT_prompt_examples +
            '\n\nQ: ' + full_question +
            "\nA: Think step by step through this question. Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions step by step and correctly. Write your final answer as: The answer is <option letter>"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()


        # === Extract the option letter
        match = re.search(r'the answer is\s*([A-E])\b', ans_model, re.IGNORECASE)
        if match:
            predicted_choice = match.group(1).upper()
        else:
            predicted_choice = None

        log_block = (
            f'Q: {full_question}\nA_model:\n{ans_model}\nExtracted Option:\n{predicted_choice}\nCorrect:\n{correct_choice}\n\n'
        )

        # === Accuracy Check
        if predicted_choice == correct_choice:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        return "error", f"Error processing entry: {d}\nException: {str(e)}\n\n"

# === Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            if result_type == "correct":
                acc += 1
                fd.write(log)
            elif result_type in ["incorrect", "error"]:
                bad_fd.write(log)
            total += 1
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")


  0%|          | 1/205 [00:42<2:25:17, 42.73s/it]

Accuracy: 1 / 1 = 100.00%
Accuracy: 1 / 2 = 50.00%


  1%|▏         | 3/205 [00:43<37:47, 11.22s/it]  

Accuracy: 2 / 3 = 66.67%


  2%|▏         | 4/205 [00:43<25:04,  7.49s/it]

...

Accuracy: 111 / 182 = 60.99%
Accuracy: 111 / 183 = 60.66%
Accuracy: 111 / 184 = 60.33%
Accuracy: 112 / 185 = 60.54%
Accuracy: 113 / 186 = 60.75%
Accuracy: 113 / 187 = 60.43%
Accuracy: 113 / 188 = 60.11%
Accuracy: 114 / 189 = 60.32%
Accuracy: 115 / 190 = 60.53%
Accuracy: 115 / 191 = 60.21%
Accuracy: 116 / 192 = 60.42%


 94%|█████████▍| 193/205 [02:51<00:02,  4.83it/s]

Accuracy: 116 / 193 = 60.10%


100%|██████████| 205/205 [02:54<00:00,  1.18it/s]

Accuracy: 117 / 194 = 60.31%
Accuracy: 118 / 195 = 60.51%
Accuracy: 119 / 196 = 60.71%
Accuracy: 119 / 197 = 60.41%
Accuracy: 120 / 198 = 60.61%
Accuracy: 120 / 199 = 60.30%
Accuracy: 120 / 200 = 60.00%
Accuracy: 121 / 201 = 60.20%
Accuracy: 122 / 202 = 60.40%
Accuracy: 123 / 203 = 60.59%
Accuracy: 123 / 204 = 60.29%
Accuracy: 124 / 205 = 60.49%


In [ ]:
import os
import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/AQuA/standard.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
os.makedirs(os.path.dirname(output_path), exist_ok=True)

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        return round(float(cleaned), 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    try:
        question = d['question']
        options = d.get('options', [])  # Optional if not MCQ
        correct_raw = d['correct'].strip()
        is_mcq = correct_raw.upper() in ['A', 'B', 'C', 'D', 'E']
        correct_value = correct_raw.upper() if is_mcq else float(correct_raw)

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}" if options else question

        # === Prompt Setup ===
        prompt_q = (
            Standard_prompt_examples +
            '\n\nQ: ' + full_question +
            "\nA: Answer the question. Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {"role": "system", "content": "Your goal is to answer these math questions correctly. Write your final answer as: The answer is <option letter>"},
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Model's Answer ===
        match = re.search(r'the answer is\s*:?\s*([A-E]|[-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        extracted = None
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            if is_mcq:
                extracted = extracted_raw.upper()
            else:
                extracted = clean_and_truncate(extracted_raw)

        # === Log Block ===
        log_block = (
            f'Q: {full_question}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted Option:\n{extracted}\n'
            f'Correct:\n{correct_raw}\n\n'
        )

        # === Check Answer ===
        if is_mcq:
            is_correct = extracted == correct_value
        else:
            is_correct = extracted is not None and math.isclose(extracted, correct_value, rel_tol=1e-4)

        if is_correct:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block

    except Exception as e:
        return "error", f"⚠️ Error processing entry: {d}\nTraceback:\n{traceback.format_exc()}\n\n"

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            total += 1
            if result_type == "correct":
                acc += 1
                fd.write(log)
            else:
                bad_fd.write(log)
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

    # === Final Summary ===
    final_summary = f"\n✅ Final Accuracy: {acc} / {total} = {acc / total:.2%}\n"
    fd.write("\n=== FINAL RESULTS ===\n" + final_summary)
    print(final_summary)


In [5]:
import os
import re
import math
import traceback
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# === Metrics ===
acc = 0
total = 0

# === File Output Paths ===
output_path = '/Users/shashanklahoti/Desktop/Coding Projects/Research_LLM_2025/Chain-of-Contrastive-Thought/research/logs/AQuA/complexCoT.txt'
bad_output_path = output_path.replace('.txt', '_bad.txt')
error_log_path = output_path.replace('.txt', '_errors.txt')

# === Cleaning & Truncation Utility ===
def clean_and_truncate(value_str):
    """Remove $, %, commas, etc. and round to 4 decimal places"""
    cleaned = re.sub(r'[^\d\.\-]', '', value_str)
    try:
        num = float(cleaned)
        return round(num, 4)
    except ValueError:
        return None

# === Function to Process a Single Entry ===
def process_entry(d):
    global acc, total
    try:
        question = d['question']
        options = d.get('options', [])  # Optional if not MCQ
        correct_raw = d['correct'].strip()
        is_mcq = correct_raw.upper() in ['A', 'B', 'C', 'D', 'E']
        correct_value = correct_raw.upper() if is_mcq else float(correct_raw)

        formatted_options = "\n".join(options)
        full_question = f"{question}\nOptions:\n{formatted_options}" if options else question


        # === Prompt Setup for Complex CoT ===
        prompt_q = (
            CCoT_prompt_examples +
            '\n\nQ: ' + full_question +
            "Please reason through this problem using a complex, multi-step chain of thought:\n"
            "Step 1: Clearly state all given information and any assumptions.\n"
            "Step 2: Propose two different methods to solve the problem, briefly outlining the logic of each.\n"
            "Step 3: For each method, work through all intermediate steps in detail, showing calculations, checks, and potential pitfalls.\n"
            "Step 4: Evaluate and compare the two methods—discussing which is better based on clarity, reliability, or efficiency.\n"
            "Step 5: Choose the better method and use it to solve the problem, showing all steps.\n"
            "Step 6: Double-check the solution for errors or unreasonable results.\n"
            "Choose the best option and write your answer as: The answer is <option letter>"
        )

        messages = [
            {
                "role": "system",
                "content": (
                    "Your goal is to answer the question using a complex, coherent, step by step thoughts, answering the questions correctly. Write your final answer as: The answer is <option letter>\n"
                )
            },
            {"role": "user", "content": prompt_q}
        ]

        response = completion_with_backoff(messages)
        ans_model = response.choices[0].message.content.strip()

        # === Extract Model's Answer ===
        match = re.search(r'the answer is\s*:?\s*([A-E]|[-\d\.,\$\%]+)', ans_model, re.IGNORECASE)
        extracted = None
        if match:
            extracted_raw = match.group(1).strip().rstrip('.')
            if is_mcq:
                extracted = extracted_raw.upper()
            else:
                extracted = clean_and_truncate(extracted_raw)

        print(extracted)

        # === Log Block ===
        log_block = (
            f'Q: {full_question}\n'
            f'A_model:\n{ans_model}\n'
            f'Extracted Option:\n{extracted}\n'
            f'Correct:\n{correct_raw}\n\n'
        )

        # === Check Answer ===
        if is_mcq:
            is_correct = extracted == correct_value
        else:
            is_correct = extracted is not None and math.isclose(extracted, correct_value, rel_tol=1e-4)

        if is_correct:
            return "correct", log_block
        else:
            return "incorrect", "❌ INCORRECT OR INVALID\n" + log_block


    except Exception as e:
        # Log the error and the problematic entry
        error_log = f"Error at index {idx}:\nData: {d}\nTraceback:\n{traceback.format_exc()}\n\n"
        return "error", error_log

# === Main Parallel Processing ===
with open(output_path, 'w') as fd, open(bad_output_path, 'w') as bad_fd:
    with ThreadPoolExecutor() as executor:
        futures = [executor.submit(process_entry, d) for d in dev_data]
        for future in tqdm(futures):
            result_type, log = future.result()
            total += 1
            if result_type == "correct":
                acc += 1
                fd.write(log)
            else:
                bad_fd.write(log)
            print(f"Accuracy: {acc} / {total} = {acc / total:.2%}")

    # === Final Summary ===
    final_summary = f"\n✅ Final Accuracy: {acc} / {total} = {acc / total:.2%}\n"
    fd.write("\n=== FINAL RESULTS ===\n" + final_summary)
    print(final_summary)

  0%|          | 0/205 [00:00<?, ?it/s]

C
B
D
A
A
D


  0%|          | 1/205 [00:05<19:31,  5.75s/it]

A
9996
A
Accuracy: 0 / 1 = 0.00%
Accuracy: 0 / 2 = 0.00%


  1%|▏         | 3/205 [00:05<05:08,  1.53s/it]

...

D
Accuracy: 112 / 188 = 59.57%
Accuracy: 113 / 189 = 59.79%
Accuracy: 114 / 190 = 60.00%
Accuracy: 115 / 191 = 60.21%
Accuracy: 116 / 192 = 60.42%
Accuracy: 116 / 193 = 60.10%
D
B
B
E
A
A


 95%|█████████▍| 194/205 [05:12<00:25,  2.34s/it]

B
Accuracy: 117 / 194 = 60.31%
Accuracy: 118 / 195 = 60.51%
Accuracy: 118 / 196 = 60.20%
Accuracy: 119 / 197 = 60.41%
Accuracy: 120 / 198 = 60.61%


100%|██████████| 205/205 [05:13<00:00,  1.53s/it]

D
Accuracy: 120 / 199 = 60.30%
Accuracy: 120 / 200 = 60.00%
Accuracy: 121 / 201 = 60.20%
Accuracy: 122 / 202 = 60.40%
Accuracy: 123 / 203 = 60.59%
Accuracy: 123 / 204 = 60.29%
Accuracy: 123 / 205 = 60.00%

✅ Final Accuracy: 123 / 205 = 60.00%

